## Tool Calling

In [ ]:
import dotenv
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

Create a function tools for carbon calculation:

In [ ]:
@function_tool
def get_scope1_emissions(emission_source: str) -> str:
    """
    Get emission factor information for Scope 1 direct emission sources in Bitcoin mining operations.
    
    Args:
        emission_source: Type of emission source (e.g., "natural_gas", "diesel", "refrigerant_r410a")
    
    Returns:
        Emission factor and calculation guidance per standard unit
    """
    # Scope 1 emission factors database
    scope1_data = {
        "natural_gas": "0.184 kgCO2e per kWh or 1.93 kgCO2e per cubic meter (for power generation)",
        "diesel": "2.68 kgCO2e per liter (for backup generators)",
        "propane": "1.51 kgCO2e per liter or 2.98 kgCO2e per kg",
        "gasoline": "2.31 kgCO2e per liter (for company vehicles)",
        "refrigerant_r410a": "2,088 kgCO2e per kg leaked (GWP)",
        "refrigerant_r134a": "1,430 kgCO2e per kg leaked (GWP)",
        "sf6": "23,500 kgCO2e per kg leaked (electrical switchgear)",
        "hfc_leak": "Varies by type: R-410A (2,088), R-134a (1,430), R-404A (3,922) kgCO2e per kg",
    }
    
    source_key = emission_source.lower()
    if source_key in scope1_data:
        return f"{emission_source.replace('_', ' ').title()}: {scope1_data[source_key]}"
    else:
        return f"I don't have emission factor data for {emission_source}. Try: natural_gas, diesel, refrigerant_r410a, or sf6."


In [ ]:
carbon_accounting_agent = Agent(
    name = "Carbon Accounting Expert Assistant",
    instructions="""
    You are a helpful assistant specializing in carbon accounting for Bitcoin mining operations.
    You provide concise, accurate answers using emission factors and calculation tools.
    When asked about emissions, use the available tools to provide specific data and calculations.
    Always cite emission factors with their units (kgCO2e per unit).
    """,
    tools=[get_scope1_emissions])

In [6]:
# Example usage
with trace("Carbon Accounting Expert Assistant"):
    result = await Runner.run(
        carbon_accounting_agent,
        "how much emission factor from the natural gas and diesel?"
    )
print(result)

RunResult:
- Last agent: Agent(name="Carbon Accounting Expert Assistant", ...)
- Final output (str):
    Here are the Scope 1 emission factors (kg CO2e per unit):
    
    - Natural gas: 0.184 kgCO2e per kWh or 1.93 kgCO2e per cubic meter
    - Diesel: 2.68 kgCO2e per liter
    
    These factors are for power generation (natural gas) and backup generators (diesel).
- 7 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [7]:
@function_tool
def get_scope2_electricity_emissions(grid_region: str) -> str:
    """
    Get grid emission factors for Scope 2 purchased electricity in different regions/countries.
    
    Args:
        grid_region: Region or country code (e.g., "indonesia", "texas", "iceland", "china")
    
    Returns:
        Grid emission factor in kgCO2e per kWh for the specified region
    """
    # Grid emission factors by region (approximate values)
    grid_factors = {
        "indonesia": "0.85 kgCO2e per kWh (coal-dominant grid, Java-Bali system)",
        "texas": "0.42 kgCO2e per kWh (ERCOT grid, natural gas dominant)",
        "iceland": "0.01 kgCO2e per kWh (geothermal and hydro)",
        "china": "0.58 kgCO2e per kWh (national average, coal-heavy)",
        "norway": "0.02 kgCO2e per kWh (hydropower dominant)",
        "kazakhstan": "0.65 kgCO2e per kWh (coal-dominant)",
        "usa_average": "0.39 kgCO2e per kWh (US national average)",
        "canada": "0.12 kgCO2e per kWh (hydro-dominant)",
        "mongolia": "0.95 kgCO2e per kWh (coal-dominant)",
        "georgia": "0.08 kgCO2e per kWh (hydropower dominant)",
    }
    
    region_key = grid_region.lower().replace(" ", "_")
    if region_key in grid_factors:
        return f"{grid_region.title()} Grid: {grid_factors[region_key]}"
    else:
        return f"I don't have grid emission data for {grid_region}. Try: indonesia, texas, iceland, china, or usa_average."


### Note on Calcuation of Mining Emissions: 
- Hashrate = total computations per second (TH/s).
- Power = electrical consumption required to deliver that hashrate (Watts).

S21: Hashrate = 200 TH/s, Power = 3,500 W, Efficiency = 17.5 J/TH

In [9]:
@function_tool
def calculate_mining_emissions(hash_rate_th: float, power_efficiency_j_th: float, 
                               grid_factor_kg_kwh: float, operating_hours: int = 8760) -> str:
    """
    Calculate annual CO2 emissions from Bitcoin mining operations.
    
    Args:
        hash_rate_th: Mining hash rate in TH/s (terahashes per second)
        power_efficiency_j_th: ASIC efficiency in J/TH (joules per terahash)
        grid_factor_kg_kwh: Grid emission factor in kgCO2e per kWh
        operating_hours: Annual operating hours (default 8760 for 24/7 operation)
    
    Returns:
        Detailed breakdown of electricity consumption and CO2 emissions
    """
    # Calculate power consumption
    power_kw = (hash_rate_th * power_efficiency_j_th) / 1000  # Convert J/s to kW
    
    # Calculate annual energy consumption
    annual_kwh = power_kw * operating_hours
    
    # Calculate emissions
    annual_co2_kg = annual_kwh * grid_factor_kg_kwh
    annual_co2_tonnes = annual_co2_kg / 1000
    
    return f"""Mining Emissions Calculation:
    - Hash Rate: {hash_rate_th:,.1f} TH/s
    - Power Efficiency: {power_efficiency_j_th} J/TH
    - Power Consumption: {power_kw:,.2f} kW
    - Annual Energy: {annual_kwh:,.0f} kWh
    - Grid Emission Factor: {grid_factor_kg_kwh} kgCO2e/kWh
    - Annual CO2 Emissions: {annual_co2_tonnes:,.2f} tonnes CO2e
    - Monthly CO2 Emissions: {annual_co2_tonnes/12:,.2f} tonnes CO2e"""

## Note on Efficiency:

- Efficiency (J/TH) = how many joules of energy are used to produce 1 TH of hashing.
- Lower efficiency (e.g., 15 J/TH) = more efficient chip → less power used.

If two miners have the same hashrate, the one with higher efficiency number = worse efficiency = more power.

Example:

| Miner     | Hashrate | Efficiency | Power   |
| --------- | -------- | ---------- | ------- |
| S21       | 200 TH/s | 17.5 J/TH  | 3,500 W |
| S21 Hydro | 335 TH/s | 16 J/TH    | 5,360 W |

Lower efficiency → you use less energy to produce the same TH/s.

-------------------

Efficiency limits the achievable hashrate per watt.

A miner cannot increase hashrate without increasing power unless its efficiency improves.

Think of:

- Higher hashrate requires more power
- But improved efficiency allows more hashrate per watt

If two devices have the same power budget (e.g., 3,500 W):
| Efficiency | Expected Hashrate |
| ---------- | ----------------- |
| 20 J/TH    | 175 TH/s          |
| 17.5 J/TH  | 200 TH/s          |
| 15 J/TH    | 233 TH/s          |

So, at fixed power, better chip efficiency = more hashrate.

In [10]:
@function_tool
def get_asic_specifications(asic_model: str) -> str:
    """
    Get technical specifications and emission-relevant data for common ASIC miners.
    
    Args:
        asic_model: ASIC miner model name (e.g., "s19_xp", "m50s", "s21")
    
    Returns:
        Hash rate, power consumption, and efficiency specifications
    """
    # Common ASIC specifications
    asic_specs = {
        "s19_xp": "Hash Rate: 140 TH/s, Power: 3,010W, Efficiency: 21.5 J/TH (Antminer S19 XP)",
        "s19j_pro": "Hash Rate: 104 TH/s, Power: 3,068W, Efficiency: 29.5 J/TH (Antminer S19j Pro)",
        "s21": "Hash Rate: 200 TH/s, Power: 3,500W, Efficiency: 17.5 J/TH (Antminer S21)",
        "m50s": "Hash Rate: 126 TH/s, Power: 3,276W, Efficiency: 26 J/TH (WhatsMiner M50S)",
        "m53s": "Hash Rate: 226 TH/s, Power: 5,772W, Efficiency: 25.5 J/TH (WhatsMiner M53S)",
        "m60s": "Hash Rate: 186 TH/s, Power: 3,344W, Efficiency: 18 J/TH (WhatsMiner M60S)",
        "avalon_1346": "Hash Rate: 130 TH/s, Power: 3,420W, Efficiency: 26.3 J/TH (AvalonMiner 1346)",
    }
    
    model_key = asic_model.lower().replace(" ", "_")
    if model_key in asic_specs:
        return f"{asic_model.upper()}: {asic_specs[model_key]}"
    else:
        return f"I don't have specifications for {asic_model}. Try: s19_xp, s21, m50s, m60s, or avalon_1346."


## Note on Power Usage Effectiveness (PUE)
PUE (Power Usage Effectiveness) = measure of datacenter overhead efficiency.

PUE = Total Facility Power / IT Power (Miners only) 

Where:

- IT = miners (ASICs)
- Overhead = cooling, fans, transformers, cables, etc.

Example:

If 1 miner consumes 3,500 W, and cooling overhead adds 700 W, then:

PUE = 3500 + 700 / 3500 = 1.20

Meaning:
20% extra power is required to run the site.

Lower PUE = more efficient mining operation

| PUE       | Meaning                              |
| --------- | ------------------------------------ |
| 1.05      | Excellent immersion cooling facility |
| 1.10–1.20 | Standard air-cooled farm             |
| 1.30–1.60 | Poor airflow design                  |


In [15]:
@function_tool
def calculate_pue_impact(mining_power_kw: float, cooling_power_kw: float, 
                        other_facility_kw: float = 0) -> str:
    """
    Calculate Power Usage Effectiveness (PUE) and its impact on total emissions.
    
    Args:
        mining_power_kw: Power consumed by ASIC miners in kW
        cooling_power_kw: Power consumed by cooling systems in kW
        other_facility_kw: Power for lighting, security, etc. in kW (default 0)
    
    Returns:
        PUE calculation and efficiency assessment
    """
    total_power_kw = mining_power_kw + cooling_power_kw + other_facility_kw
    pue = total_power_kw / mining_power_kw if mining_power_kw > 0 else 0
    
    # Assessment
    if pue <= 1.1:
        assessment = "Excellent (world-class efficiency)"
    elif pue <= 1.2:
        assessment = "Very Good (immersion/optimized cooling)"
    elif pue <= 1.5:
        assessment = "Good (efficient air cooling)"
    elif pue <= 2.0:
        assessment = "Fair (standard operation)"
    else:
        assessment = "Poor (needs optimization)"
    
    overhead_percentage = ((pue - 1.0) * 100) if pue > 0 else 0
    
    return f"""PUE Analysis:
    - ASIC Mining Power: {mining_power_kw:,.2f} kW
    - Cooling Power: {cooling_power_kw:,.2f} kW
    - Other Facility Power: {other_facility_kw:,.2f} kW
    - Total Facility Power: {total_power_kw:,.2f} kW
    - PUE Ratio: {pue:.3f}
    - Assessment: {assessment}
    - Overhead: {overhead_percentage:.1f}% additional emissions beyond mining
    
    Note: Lower PUE means better efficiency. Target PUE < 1.2 for mining operations."""


In [16]:
@function_tool
def get_renewable_energy_factor(renewable_type: str) -> str:
    """
    Get emission factors for different renewable energy sources and considerations.
    
    Args:
        renewable_type: Type of renewable energy (e.g., "solar", "wind", "hydro", "geothermal")
    
    Returns:
        Lifecycle emission factor and relevant characteristics
    """
    renewable_factors = {
        "solar": "0.041 kgCO2e per kWh (lifecycle), includes panel manufacturing and installation",
        "wind": "0.011 kgCO2e per kWh (lifecycle), includes turbine manufacturing",
        "hydro": "0.024 kgCO2e per kWh (lifecycle), varies by reservoir size",
        "geothermal": "0.038 kgCO2e per kWh (lifecycle), minimal operational emissions",
        "nuclear": "0.012 kgCO2e per kWh (lifecycle), includes construction and decommissioning",
        "biomass": "0.230 kgCO2e per kWh (lifecycle), carbon neutral if sustainably sourced",
        "flare_gas": "0.56 kgCO2e per kWh (combustion only), but avoids methane release (counterfactual benefit)",
    }
    
    renewable_key = renewable_type.lower().replace(" ", "_")
    if renewable_key in renewable_factors:
        return f"{renewable_type.title()}: {renewable_factors[renewable_key]}"
    else:
        return f"I don't have data for {renewable_type}. Try: solar, wind, hydro, geothermal, or flare_gas."

In [14]:
# Create agent with tools
carbon_accounting_agent = Agent(
    name="Carbon Accounting Expert Assistant",
    instructions="""
    You are a helpful assistant specializing in carbon accounting for Bitcoin mining operations.
    You provide concise, accurate answers using emission factors and calculation tools.
    When asked about emissions, use the available tools to provide specific data and calculations.
    Always cite emission factors with their units (kgCO2e per unit).
    """,
    tools=[
        get_scope1_emissions,
        get_scope2_electricity_emissions,
        calculate_mining_emissions,
        get_asic_specifications,
        calculate_pue_impact,
        get_renewable_energy_factor,
    ]
)

# Example usage
with trace("Carbon Accounting Expert Assistant"):
    result = await Runner.run(
        carbon_accounting_agent,
        "Calculate the annual emissions for a mining operation in Indonesia with 100 Antminer S19 XP units running 24/7 with a PUE of 1.15"
    )
print(result)

RunResult:
- Last agent: Agent(name="Carbon Accounting Expert Assistant", ...)
- Final output (str):
    Here are the results with PUE 1.15 included.
    
    - ASIC: Antminer S19 XP
      - Hash rate: 140 TH/s
      - Power efficiency: 21.5 J/TH
    
    - Fleet: 100 units
      - Total hash rate: 14,000 TH/s
      - Power consumption (ASIC): ~301 kW
      - Annual energy (base, PUE=1.0): ~2,636,760 kWh
    
    - Indonesia grid factor (coal-dominant): 0.85 kgCO2e/kWh
    
    - With PUE 1.15:
      - Annual CO2e emissions: ~2,577 tonnes CO2e per year
      - Monthly CO2e emissions: ~214.8 tonnes CO2e per month
    
    Notes:
    - The base calculation (PUE=1.0) yields 2,241.25 tonnes CO2e/year; applying 1.15 PUE gives 2,577.4 tonnes CO2e/year.
    - Grid factor cited: 0.85 kgCO2e per kWh.
- 9 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Example Queries the Agent Can Now Handle

1. "What's the emission factor for diesel used in backup generators?"
2. "Calculate emissions for 1000 TH/s mining operation in Texas"
3. "What are the specifications of the Antminer S21?"
4. "Compare grid emission factors between Indonesia and Iceland"
5. "What's the PUE impact if my cooling uses 200kW and mining uses 1000kW?"
6. "What's the lifecycle emissions of solar power for mining?"
7. "How much CO2 from refrigerant R-410A leak of 5kg?"